# 30. 再構築データでのモデル再評価
出典：FX (3).ipynb、元セルindex [66, 67]。保存出力は results/imported_fx3/。
研究履歴の原本です。Notebookの変数・価格CSV・学習済みファイルに依存します。
失敗した試行も保管しています。一括実行やAPI接続を開始する入口ではありません。
元コード内の指示・自動判定名は資料として保存しています。独立した検証済みの結論とは区別してください。


## 元セルindex 66
構文状態：valid


In [ ]:
# ============================================================
# CLEAN DATA - HGB BASE FULL REVALIDATION
#
# 正式なclean historical dataset上でBASE Pipelineをゼロから再評価
#
# Pipeline:
# USDJPY 15m
# -> BASE 30 features
# -> HistGradientBoosting
# -> Calibration
# -> Confidence
# -> Threshold
# -> Session
# -> Position Sizing
# -> Fixed 30m Exit
# -> Cost
# -> Nested OOS
#
# Development : 2020-2025
# Confirmation: 2026
# ============================================================

import math
import warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 200)


# ============================================================
# 0. CONFIG
# ============================================================

RANDOM_STATE = 42

BASE_COST = 0.00004
# 0.004%

DEVELOPMENT_YEARS = [
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
]

CONFIRMATION_YEAR = 2026


THRESHOLDS = [
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
]


SESSIONS = [
    "ALL",
    "UTC_13_24",
    "UTC_21_24",
    "EXCLUDE_08_13",
]


CALIBRATION_METHODS = [
    "RAW",
    "PLATT",
    "ISOTONIC",
]


SIZING_POLICIES = [
    "FIXED",
    "GENTLE",
    "MODERATE",
    "STRONG",
]


HGB_CONFIG = {

    "learning_rate":
        0.05,

    "max_iter":
        250,

    "max_leaf_nodes":
        15,

    "min_samples_leaf":
        30,

    "l2_regularization":
        1.0,

    "early_stopping":
        False,

    "random_state":
        RANDOM_STATE,
}


MIN_TRAIN_ROWS = 10000
MIN_VALIDATION_ROWS = 500
MIN_TEST_ROWS = 500

MIN_VALIDATION_TRADES = 40


# ============================================================
# 1. CLEAN bars VALIDATION
# ============================================================

if "bars" not in globals():

    raise RuntimeError(
        "`bars` がありません。"
    )


if not isinstance(
    bars,
    pd.DataFrame
):

    raise TypeError(
        "`bars` はDataFrameである必要があります。"
    )


BARS = bars.copy()


BARS.columns = [

    str(c)
    .strip()
    .lower()

    for c in BARS.columns
]


required_ohlc = [
    "open",
    "high",
    "low",
    "close",
]


missing = [

    c
    for c in required_ohlc

    if c not in BARS.columns
]


if missing:

    raise RuntimeError(
        f"OHLC列不足: {missing}"
    )


BARS = BARS[
    required_ohlc
].copy()


if not isinstance(
    BARS.index,
    pd.DatetimeIndex
):

    raise RuntimeError(
        "bars.index がDatetimeIndexではありません。"
    )


BARS.index = pd.to_datetime(

    BARS.index,

    utc=True,

    errors="coerce"
)


BARS = BARS.loc[
    ~BARS.index.isna()
].copy()


BARS = BARS.sort_index()


# ------------------------------------------------------------
# duplicate
# ------------------------------------------------------------

duplicate_count = int(
    BARS.index
    .duplicated()
    .sum()
)


if duplicate_count:

    raise RuntimeError(
        f"duplicate timestamp: {duplicate_count}"
    )


# ------------------------------------------------------------
# 15m grid
# ------------------------------------------------------------

bad_grid = (

    (BARS.index.minute % 15 != 0)

    |

    (BARS.index.second != 0)

    |

    (BARS.index.microsecond != 0)
)


bad_grid_count = int(
    np.asarray(
        bad_grid
    ).sum()
)


if bad_grid_count:

    raise RuntimeError(
        f"15分grid外timestamp: {bad_grid_count}"
    )


# ------------------------------------------------------------
# numeric
# ------------------------------------------------------------

for c in required_ohlc:

    BARS[c] = pd.to_numeric(
        BARS[c],
        errors="coerce"
    )


if BARS[
    required_ohlc
].isna().any().any():

    raise RuntimeError(
        "OHLCにNaNがあります。"
    )


# ------------------------------------------------------------
# OHLC validity
# ------------------------------------------------------------

invalid_high = (

    BARS["high"]

    <

    BARS[
        [
            "open",
            "low",
            "close",
        ]
    ].max(axis=1)
)


invalid_low = (

    BARS["low"]

    >

    BARS[
        [
            "open",
            "high",
            "close",
        ]
    ].min(axis=1)
)


if (
    invalid_high.any()
    or
    invalid_low.any()
):

    raise RuntimeError(
        "OHLC整合性エラーがあります。"
    )


# ============================================================
# 2. CLEAN DATAの巨大jumpを再確認
# ============================================================

time_series = pd.Series(
    BARS.index,
    index=BARS.index
)


prev_time = (
    time_series.shift(1)
)


prev_close = (
    BARS["close"].shift(1)
)


contiguous_prev = (

    (
        time_series
        -
        prev_time
    )

    ==

    pd.Timedelta(
        minutes=15
    )
)


open_gap = pd.Series(

    np.where(

        contiguous_prev,

        BARS["open"]
        /
        prev_close
        -
        1,

        np.nan
    ),

    index=BARS.index
)


extreme_5pct_gap = (

    open_gap.abs()
    >
    0.05
)


extreme_5pct_count = int(
    extreme_5pct_gap.sum()
)


if extreme_5pct_count > 0:

    raise RuntimeError(

        "Clean barsに連続15分で"
        f">5% gapが {extreme_5pct_count} 件あります。"
    )


print("=" * 100)
print("CLEAN DATA PREFLIGHT")
print("=" * 100)

print(
    "Rows:",
    f"{len(BARS):,}"
)

print(
    "Period:",
    BARS.index.min(),
    "->",
    BARS.index.max()
)

print(
    "Duplicate timestamps:",
    duplicate_count
)

print(
    "Bad 15m timestamps:",
    bad_grid_count
)

print(
    "Contiguous >5% gaps:",
    extreme_5pct_count
)


# ============================================================
# 3. BASE 30 FEATURES
# ============================================================

BASE_FEATURES = [

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",

    "weekday",
]


if len(BASE_FEATURES) != 30:

    raise RuntimeError(
        f"BASE features != 30: {len(BASE_FEATURES)}"
    )


# ============================================================
# 4. RSI
# ============================================================

def calc_rsi(
    close,
    period=14
):

    delta = close.diff()

    gain = delta.clip(
        lower=0
    )

    loss = (
        -delta.clip(
            upper=0
        )
    )


    avg_gain = gain.rolling(
        period
    ).mean()


    avg_loss = loss.rolling(
        period
    ).mean()


    rs = (

        avg_gain

        /

        avg_loss.replace(
            0,
            np.nan
        )
    )


    return (

        100

        -

        (
            100
            /
            (
                1 + rs
            )
        )
    )


# ============================================================
# 5. FEATURE ENGINEERING
# ============================================================

def make_base_features(
    bars_input
):

    x = bars_input.copy()


    # --------------------------------------------------------
    # returns
    # --------------------------------------------------------

    for n in [
        1,
        2,
        4,
        8,
        16,
    ]:

        x[
            f"return_{n}"
        ] = (

            x["close"]
            .pct_change(
                n
            )
        )


    # --------------------------------------------------------
    # volatility
    # --------------------------------------------------------

    for n in [
        4,
        8,
        16,
        32,
    ]:

        x[
            f"vol_{n}"
        ] = (

            x["return_1"]
            .rolling(
                n
            )
            .std()
        )


    # --------------------------------------------------------
    # moving averages
    # --------------------------------------------------------

    for p in [
        5,
        10,
        20,
        50,
        100,
    ]:

        ma = (

            x["close"]
            .rolling(
                p
            )
            .mean()
        )


        x[
            f"ma{p}_distance"
        ] = (

            x["close"]
            /
            ma
            -
            1
        )


        x[
            f"ma{p}_slope"
        ] = (

            ma.pct_change()
        )


    # --------------------------------------------------------
    # candle features
    # --------------------------------------------------------

    candle_range = (

        x["high"]
        -
        x["low"]

    ).replace(
        0,
        np.nan
    )


    x["body"] = (

        x["close"]
        -
        x["open"]

    ) / candle_range


    x["upper_wick"] = (

        x["high"]

        -

        x[
            [
                "open",
                "close"
            ]
        ].max(axis=1)

    ) / candle_range


    x["lower_wick"] = (

        x[
            [
                "open",
                "close"
            ]
        ].min(axis=1)

        -

        x["low"]

    ) / candle_range


    x["range_pct"] = (

        x["high"]
        -
        x["low"]

    ) / x["close"]


    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    x["rsi14"] = (

        calc_rsi(
            x["close"],
            14
        )

        /

        100.0
    )


    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------

    prev_close = (

        x["close"]
        .shift(1)
    )


    tr = pd.concat(

        [

            x["high"]
            -
            x["low"],

            (
                x["high"]
                -
                prev_close
            ).abs(),

            (
                x["low"]
                -
                prev_close
            ).abs(),
        ],

        axis=1

    ).max(axis=1)


    atr14_abs = (

        tr
        .rolling(
            14
        )
        .mean()
    )


    x["atr14"] = (

        atr14_abs
        /
        x["close"]
    )


    # --------------------------------------------------------
    # high / low distance
    # --------------------------------------------------------

    high16 = (

        x["high"]
        .rolling(
            16
        )
        .max()
    )


    low16 = (

        x["low"]
        .rolling(
            16
        )
        .min()
    )


    x["distance_high_16"] = (

        high16
        -
        x["close"]

    ) / x["close"]


    x["distance_low_16"] = (

        x["close"]
        -
        low16

    ) / x["close"]


    # --------------------------------------------------------
    # time
    # --------------------------------------------------------

    hour = (

        x.index.hour

        +

        x.index.minute
        /
        60.0
    )


    x["hour_sin"] = np.sin(

        2
        *
        np.pi
        *
        hour
        /
        24
    )


    x["hour_cos"] = np.cos(

        2
        *
        np.pi
        *
        hour
        /
        24
    )


    x["weekday"] = (

        x.index.dayofweek

        /

        4.0
    )


    return x.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )


# ============================================================
# 6. DATASET + 30m LABEL
#
# signal = bar t close
# entry  = Open(t+1)
# exit   = Close(t+2)
#
# Entry時刻からExit bar closeまで30分
# ============================================================

def prepare_dataset(
    bars_input
):

    x = make_base_features(
        bars_input
    )


    ts = pd.Series(
        x.index,
        index=x.index
    )


    x["entry_time"] = (
        ts.shift(-1)
    )


    x["exit_bar_time"] = (
        ts.shift(-2)
    )


    x["label_end"] = (

        x["exit_bar_time"]

        +

        pd.Timedelta(
            minutes=15
        )
    )


    x["entry_price"] = (

        bars_input[
            "open"
        ]
        .shift(-1)
    )


    x["exit_price"] = (

        bars_input[
            "close"
        ]
        .shift(-2)
    )


    x["future_return"] = (

        x["exit_price"]

        /

        x["entry_price"]

        -

        1
    )


    x["target"] = (

        x["future_return"]
        >
        0

    ).astype(int)


    # --------------------------------------------------------
    # 必ず連続barのみ使用
    # --------------------------------------------------------

    continuous = (

        (
            x["entry_time"]
            -
            ts
        )

        ==

        pd.Timedelta(
            minutes=15
        )

    ) & (

        (
            x["exit_bar_time"]
            -
            ts
        )

        ==

        pd.Timedelta(
            minutes=30
        )
    )


    required = (

        BASE_FEATURES

        +

        [
            "entry_time",
            "exit_bar_time",
            "label_end",
            "entry_price",
            "exit_price",
            "future_return",
            "target",
        ]
    )


    out = (

        x.loc[
            continuous
        ]

        .dropna(
            subset=required
        )

        .copy()
    )


    return out


DATA = prepare_dataset(
    BARS
)


print()
print("=" * 100)
print("MODEL DATASET")
print("=" * 100)

print(
    "Usable rows:",
    f"{len(DATA):,}"
)

print(
    "Features:",
    len(BASE_FEATURES)
)


# ============================================================
# 7. PERFORMANCE METRICS
# ============================================================

def stats_of_returns(
    returns
):

    r = np.asarray(
        returns,
        dtype=float
    )


    r = r[
        np.isfinite(r)
    ]


    if len(r) == 0:

        return {

            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "profit_factor":
                np.nan,

            "growth":
                0.0,

            "max_dd":
                np.nan,

            "return_to_dd":
                np.nan,
        }


    if np.any(
        r <= -1
    ):

        raise RuntimeError(
            "return <= -100% が存在します。"
        )


    gains = (

        r[
            r > 0
        ]
        .sum()
    )


    losses = (

        -r[
            r < 0
        ]
        .sum()
    )


    if losses > 0:

        pf = (

            gains
            /
            losses
        )


    elif gains > 0:

        pf = np.inf


    else:

        pf = np.nan


    equity = np.r_[

        1.0,

        np.cumprod(
            1 + r
        )
    ]


    peak = np.maximum.accumulate(
        equity
    )


    dd = (

        equity
        /
        peak
        -
        1
    )


    max_dd = float(
        dd.min()
    )


    growth = float(
        equity[-1]
        -
        1
    )


    if max_dd < 0:

        return_to_dd = (

            growth
            /
            abs(
                max_dd
            )
        )


    else:

        return_to_dd = np.nan


    return {

        "trades":
            int(
                len(r)
            ),

        "win_rate":
            float(
                (
                    r > 0
                ).mean()
            ),

        "avg_return":
            float(
                r.mean()
            ),

        "profit_factor":
            float(
                pf
            ),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            float(
                return_to_dd
            ),
    }


# ============================================================
# 8. ECE
# ============================================================

def expected_calibration_error(
    y_true,
    probability,
    bins=10
):

    y = np.asarray(
        y_true
    )

    p = np.asarray(
        probability
    )


    edges = np.linspace(
        0,
        1,
        bins + 1
    )


    ece = 0.0


    for i in range(
        bins
    ):

        if i == bins - 1:

            mask = (

                (p >= edges[i])

                &

                (p <= edges[i + 1])
            )

        else:

            mask = (

                (p >= edges[i])

                &

                (p < edges[i + 1])
            )


        n = int(
            mask.sum()
        )


        if n == 0:

            continue


        actual = float(
            y[
                mask
            ].mean()
        )


        predicted = float(
            p[
                mask
            ].mean()
        )


        ece += (

            n
            /
            len(y)

            *

            abs(
                actual
                -
                predicted
            )
        )


    return float(
        ece
    )


# ============================================================
# 9. SPLIT
# ============================================================

def make_split(
    dataset,
    test_year
):

    validation_year = (
        test_year
        -
        1
    )


    validation_start = pd.Timestamp(
        f"{validation_year}-01-01",
        tz="UTC"
    )


    test_start = pd.Timestamp(
        f"{test_year}-01-01",
        tz="UTC"
    )


    test_end = pd.Timestamp(
        f"{test_year + 1}-01-01",
        tz="UTC"
    )


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    train = dataset.loc[

        (
            dataset.index
            <
            validation_start
        )

        &

        (
            dataset["label_end"]
            <=
            validation_start
        )

    ].copy()


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    validation = dataset.loc[

        (
            dataset.index
            >=
            validation_start
        )

        &

        (
            dataset.index
            <
            test_start
        )

        &

        (
            dataset["label_end"]
            <=
            test_start
        )

    ].copy()


    # --------------------------------------------------------
    # Final training = Train + Validation period
    # --------------------------------------------------------

    final_train = dataset.loc[

        (
            dataset.index
            <
            test_start
        )

        &

        (
            dataset["label_end"]
            <=
            test_start
        )

    ].copy()


    # --------------------------------------------------------
    # Test
    # --------------------------------------------------------

    test = dataset.loc[

        (
            dataset.index
            >=
            test_start
        )

        &

        (
            dataset.index
            <
            test_end
        )

        &

        (
            dataset["label_end"]
            <=
            test_end
        )

    ].copy()


    if (
        len(train)
        <
        MIN_TRAIN_ROWS
    ):

        return None


    if (
        len(validation)
        <
        MIN_VALIDATION_ROWS
    ):

        return None


    if (
        len(test)
        <
        MIN_TEST_ROWS
    ):

        return None


    return {

        "train":
            train,

        "validation":
            validation,

        "final_train":
            final_train,

        "test":
            test,
    }


# ============================================================
# 10. HGB
# ============================================================

def fit_hgb(
    frame
):

    model = HistGradientBoostingClassifier(
        **HGB_CONFIG
    )


    model.fit(

        frame[
            BASE_FEATURES
        ],

        frame[
            "target"
        ]
    )


    return model


def model_probability(
    model,
    frame
):

    return (

        model.predict_proba(

            frame[
                BASE_FEATURES
            ]

        )[:, 1]
    )


# ============================================================
# 11. CALIBRATORS
# ============================================================

class RawCalibrator:

    def fit(
        self,
        p,
        y
    ):

        return self


    def predict(
        self,
        p
    ):

        return np.asarray(
            p,
            dtype=float
        )


class PlattCalibrator:

    def __init__(
        self
    ):

        self.model = LogisticRegression(

            solver="lbfgs",

            random_state=
                RANDOM_STATE
        )


    def fit(
        self,
        p,
        y
    ):

        y = np.asarray(
            y
        )


        if len(
            np.unique(
                y
            )
        ) < 2:

            return RawCalibrator()


        self.model.fit(

            np.asarray(
                p
            ).reshape(
                -1,
                1
            ),

            y
        )


        return self


    def predict(
        self,
        p
    ):

        return (

            self.model
            .predict_proba(

                np.asarray(
                    p
                ).reshape(
                    -1,
                    1
                )

            )[:, 1]
        )


class IsotonicCalibrator:

    def __init__(
        self
    ):

        self.model = IsotonicRegression(

            y_min=0,

            y_max=1,

            out_of_bounds="clip"
        )


    def fit(
        self,
        p,
        y
    ):

        self.model.fit(
            np.asarray(
                p
            ),
            np.asarray(
                y
            )
        )


        return self


    def predict(
        self,
        p
    ):

        return np.asarray(

            self.model.predict(
                np.asarray(
                    p
                )
            )
        )


# ============================================================
# 12. EXPANDING OOF
# ============================================================

def expanding_oof(
    frame
):

    years = sorted(
        frame.index.year.unique()
    )


    parts = []


    for year in years:

        prior_years = [

            y
            for y in years

            if y < year
        ]


        # 最低2年分過去が必要
        if len(
            prior_years
        ) < 2:

            continue


        start = pd.Timestamp(
            f"{year}-01-01",
            tz="UTC"
        )


        end = pd.Timestamp(
            f"{year+1}-01-01",
            tz="UTC"
        )


        history = frame.loc[

            (
                frame.index
                <
                start
            )

            &

            (
                frame["label_end"]
                <=
                start
            )

        ]


        oof = frame.loc[

            (
                frame.index
                >=
                start
            )

            &

            (
                frame.index
                <
                end
            )

            &

            (
                frame["label_end"]
                <=
                end
            )

        ]


        if (
            len(history)
            <
            MIN_TRAIN_ROWS
        ):

            continue


        if (
            len(oof)
            <
            MIN_VALIDATION_ROWS
        ):

            continue


        if (
            history["target"]
            .nunique()
            <
            2
        ):

            continue


        model = fit_hgb(
            history
        )


        p = model_probability(
            model,
            oof
        )


        parts.append(

            pd.DataFrame(

                {

                    "probability":
                        p,

                    "target":
                        oof[
                            "target"
                        ].values,
                },

                index=oof.index
            )
        )


    if not parts:

        return pd.DataFrame(

            columns=[
                "probability",
                "target"
            ]
        )


    return pd.concat(
        parts
    ).sort_index()


# ============================================================
# 13. FIT CALIBRATOR
# ============================================================

def fit_calibrator(
    method,
    oof
):

    if (
        method
        ==
        "RAW"
    ):

        return RawCalibrator()


    if len(
        oof
    ) < 500:

        return RawCalibrator()


    p = oof[
        "probability"
    ].to_numpy()


    y = oof[
        "target"
    ].to_numpy()


    if method == "PLATT":

        if len(
            np.unique(
                y
            )
        ) < 2:

            return RawCalibrator()


        cal = PlattCalibrator()

        result = cal.fit(
            p,
            y
        )


        return result


    if method == "ISOTONIC":

        if len(
            oof
        ) < 1000:

            return RawCalibrator()


        cal = IsotonicCalibrator()

        cal.fit(
            p,
            y
        )


        return cal


    return RawCalibrator()


# ============================================================
# 14. CALIBRATION SELECTION
# ============================================================

def choose_calibration(
    train,
    validation
):

    oof = expanding_oof(
        train
    )


    model = fit_hgb(
        train
    )


    raw_validation = (
        model_probability(
            model,
            validation
        )
    )


    rows = []


    for method in (
        CALIBRATION_METHODS
    ):

        calibrator = fit_calibrator(
            method,
            oof
        )


        p = np.clip(

            calibrator.predict(
                raw_validation
            ),

            0,
            1
        )


        brier = brier_score_loss(

            validation[
                "target"
            ],

            p
        )


        ece = expected_calibration_error(

            validation[
                "target"
            ],

            p
        )


        rows.append(

            {

                "method":
                    method,

                "brier":
                    brier,

                "ece":
                    ece,
            }
        )


    table = pd.DataFrame(
        rows
    )


    table = table.sort_values(

        [
            "brier",
            "ece",
            "method"
        ]
    )


    chosen = str(
        table.iloc[0][
            "method"
        ]
    )


    return (
        chosen,
        table
    )


# ============================================================
# 15. PREDICTION FRAME
# ============================================================

def prediction_frame(
    data,
    probability
):

    p = np.asarray(
        probability,
        dtype=float
    )


    direction = np.where(

        p
        >=
        0.5,

        1.0,

        -1.0
    )


    out = data[
        [
            "entry_time",
            "exit_bar_time",
            "label_end",
            "future_return",
            "target",
        ]
    ].copy()


    out["p_up"] = p


    out["confidence"] = np.maximum(

        p,

        1 - p
    )


    out["side"] = np.where(

        p >= 0.5,

        "BUY",

        "SELL"
    )


    out["gross_return"] = (

        out[
            "future_return"
        ].to_numpy()

        *

        direction
    )


    return out


# ============================================================
# 16. SESSION
# ============================================================

def session_mask(
    index,
    policy
):

    hour = index.hour


    if policy == "ALL":

        return np.ones(
            len(index),
            dtype=bool
        )


    if policy == "UTC_13_24":

        return (

            hour
            >=
            13
        )


    if policy == "UTC_21_24":

        return (

            hour
            >=
            21
        )


    if policy == "EXCLUDE_08_13":

        return ~(

            (
                hour
                >=
                8
            )

            &

            (
                hour
                <
                13
            )
        )


    raise ValueError(
        policy
    )


# ============================================================
# 17. TRADE SELECTION
#
# overlapを禁止
# ============================================================

def select_trades(
    prediction,
    threshold,
    session
):

    mask = (

        (
            prediction[
                "confidence"
            ]

            >=

            threshold
        )

        &

        session_mask(

            prediction.index,

            session
        )
    )


    candidates = (

        prediction.loc[
            mask
        ]

        .sort_index()
    )


    selected_index = []


    next_free_time = None


    for row in (
        candidates.itertuples()
    ):

        if (

            next_free_time
            is not None

            and

            row.entry_time
            <
            next_free_time
        ):

            continue


        selected_index.append(
            row.Index
        )


        next_free_time = (
            row.label_end
        )


    selected = candidates.loc[
        selected_index
    ].copy()


    selected[
        "base_net_return"
    ] = (

        selected[
            "gross_return"
        ]

        -

        BASE_COST
    )


    return selected


# ============================================================
# 18. THRESHOLD / SESSION SELECTION
# ============================================================

def choose_threshold_session(
    validation_prediction
):

    rows = []


    best = None
    best_key = None


    for threshold in THRESHOLDS:

        for session in SESSIONS:

            trades = select_trades(

                validation_prediction,

                threshold,

                session
            )


            stats = stats_of_returns(

                trades[
                    "base_net_return"
                ]
            )


            eligible = (

                stats[
                    "trades"
                ]

                >=

                MIN_VALIDATION_TRADES
            )


            if eligible:

                score = (

                    stats[
                        "avg_return"
                    ]

                    *

                    math.sqrt(

                        stats[
                            "trades"
                        ]
                    )
                )

            else:

                score = np.nan


            rows.append(

                {

                    "threshold":
                        threshold,

                    "session":
                        session,

                    "eligible":
                        eligible,

                    "score":
                        score,

                    **stats,
                }
            )


            if not eligible:

                continue


            pf_key = (

                stats[
                    "profit_factor"
                ]

                if np.isfinite(

                    stats[
                        "profit_factor"
                    ]

                )

                else -999
            )


            key = (

                score,

                pf_key,

                stats[
                    "trades"
                ],

                -threshold
            )


            if (

                best_key
                is None

                or

                key
                >
                best_key
            ):

                best_key = key

                best = (

                    threshold,

                    session
                )


    if best is None:

        raise RuntimeError(
            "ValidationでThreshold / Sessionを選択できません。"
        )


    return (

        best[0],

        best[1],

        pd.DataFrame(
            rows
        )
    )


# ============================================================
# 19. POSITION SIZING
# ============================================================

def raw_position_size(
    confidence,
    threshold,
    policy
):

    c = np.asarray(
        confidence,
        dtype=float
    )


    edge = (

        c
        -
        threshold

    ) / max(

        1
        -
        threshold,

        1e-12
    )


    edge = np.clip(

        edge,

        0,

        1
    )


    if policy == "FIXED":

        return np.ones_like(
            edge
        )


    if policy == "GENTLE":

        return (

            0.85

            +

            0.30
            *
            edge
        )


    if policy == "MODERATE":

        return (

            0.70

            +

            0.60
            *
            edge
        )


    if policy == "STRONG":

        return (

            0.50

            +

            1.00
            *
            edge
        )


    raise ValueError(
        policy
    )


# ============================================================
# 20. SIZING SELECTION
# ============================================================

def choose_sizing(
    validation_trades,
    threshold
):

    if len(
        validation_trades
    ) == 0:

        return (
            "FIXED",
            1.0,
            pd.DataFrame()
        )


    rows = []


    for policy in (
        SIZING_POLICIES
    ):

        raw_size = raw_position_size(

            validation_trades[
                "confidence"
            ],

            threshold,

            policy
        )


        mean_raw = raw_size.mean()


        if (
            not np.isfinite(
                mean_raw
            )

            or

            mean_raw <= 0
        ):

            continue


        # 平均exposure = 1
        scale = (

            1.0
            /
            mean_raw
        )


        size = (

            raw_size

            *

            scale
        )


        returns = (

            size

            *

            (

                validation_trades[
                    "gross_return"
                ].to_numpy()

                -

                BASE_COST
            )
        )


        stats = stats_of_returns(
            returns
        )


        rows.append(

            {

                "policy":
                    policy,

                "scale":
                    scale,

                **stats,
            }
        )


    table = pd.DataFrame(
        rows
    )


    fixed_rows = table.loc[
        table[
            "policy"
        ]
        ==
        "FIXED"
    ]


    if len(
        fixed_rows
    ) == 0:

        return (
            "FIXED",
            1.0,
            table
        )


    fixed = fixed_rows.iloc[
        0
    ]


    candidates = []


    for _, row in (
        table.iterrows()
    ):

        if (
            row[
                "policy"
            ]
            ==
            "FIXED"
        ):

            continue


        if (

            row[
                "avg_return"
            ]

            >=

            fixed[
                "avg_return"
            ]

            and

            row[
                "profit_factor"
            ]

            >=

            fixed[
                "profit_factor"
            ]

            and

            row[
                "return_to_dd"
            ]

            >=

            fixed[
                "return_to_dd"
            ]
        ):

            candidates.append(
                row
            )


    if not candidates:

        return (
            "FIXED",
            1.0,
            table
        )


    chosen = max(

        candidates,

        key=lambda row:

        (

            row[
                "return_to_dd"
            ],

            row[
                "profit_factor"
            ],

            row[
                "avg_return"
            ]
        )
    )


    return (

        str(
            chosen[
                "policy"
            ]
        ),

        float(
            chosen[
                "scale"
            ]
        ),

        table
    )


# ============================================================
# 21. APPLY SIZING
# ============================================================

def apply_sizing(
    trades,
    threshold,
    policy,
    scale,
    cost_multiplier=1.0
):

    out = trades.copy()


    size = (

        raw_position_size(

            out[
                "confidence"
            ],

            threshold,

            policy
        )

        *

        scale
    )


    size = np.clip(

        size,

        0.25,

        2.0
    )


    out[
        "position_size"
    ] = size


    out[
        "net_return"
    ] = (

        size

        *

        (

            out[
                "gross_return"
            ].to_numpy()

            -

            BASE_COST
            *
            cost_multiplier
        )
    )


    return out


# ============================================================
# 22. YEAR EVALUATION
# ============================================================

def evaluate_year(
    dataset,
    test_year
):

    split = make_split(

        dataset,

        test_year
    )


    if split is None:

        return None


    train = split[
        "train"
    ]


    validation = split[
        "validation"
    ]


    final_train = split[
        "final_train"
    ]


    test = split[
        "test"
    ]


    # --------------------------------------------------------
    # Calibration method選択
    # --------------------------------------------------------

    calibration_method, calibration_table = (
        choose_calibration(

            train,

            validation
        )
    )


    # --------------------------------------------------------
    # Validation probability
    # --------------------------------------------------------

    train_oof = expanding_oof(
        train
    )


    validation_model = fit_hgb(
        train
    )


    raw_validation = model_probability(

        validation_model,

        validation
    )


    validation_calibrator = fit_calibrator(

        calibration_method,

        train_oof
    )


    calibrated_validation = np.clip(

        validation_calibrator.predict(
            raw_validation
        ),

        0,

        1
    )


    validation_prediction = prediction_frame(

        validation,

        calibrated_validation
    )


    # --------------------------------------------------------
    # Threshold / Session selection
    # --------------------------------------------------------

    (
        threshold,
        session,
        threshold_table
    ) = choose_threshold_session(

        validation_prediction
    )


    validation_selected = select_trades(

        validation_prediction,

        threshold,

        session
    )


    # --------------------------------------------------------
    # Sizing selection
    # --------------------------------------------------------

    (
        sizing_policy,
        sizing_scale,
        sizing_table
    ) = choose_sizing(

        validation_selected,

        threshold
    )


    # --------------------------------------------------------
    # Final model
    # --------------------------------------------------------

    final_oof = expanding_oof(
        final_train
    )


    final_calibrator = fit_calibrator(

        calibration_method,

        final_oof
    )


    final_model = fit_hgb(
        final_train
    )


    raw_test = model_probability(

        final_model,

        test
    )


    calibrated_test = np.clip(

        final_calibrator.predict(
            raw_test
        ),

        0,

        1
    )


    test_prediction = prediction_frame(

        test,

        calibrated_test
    )


    selected_test = select_trades(

        test_prediction,

        threshold,

        session
    )


    final_trades = apply_sizing(

        selected_test,

        threshold,

        sizing_policy,

        sizing_scale,

        cost_multiplier=1.0
    )


    stats = stats_of_returns(

        final_trades[
            "net_return"
        ]
    )


    # --------------------------------------------------------
    # Probability quality
    # --------------------------------------------------------

    if (
        test[
            "target"
        ].nunique()
        ==
        2
    ):

        auc = roc_auc_score(

            test[
                "target"
            ],

            calibrated_test
        )

    else:

        auc = np.nan


    brier = brier_score_loss(

        test[
            "target"
        ],

        calibrated_test
    )


    ece = expected_calibration_error(

        test[
            "target"
        ],

        calibrated_test
    )


    row = {

        "test_year":
            test_year,

        "validation_year":
            test_year - 1,

        "calibration":
            calibration_method,

        "threshold":
            threshold,

        "session":
            session,

        "sizing":
            sizing_policy,

        "auc":
            auc,

        "brier":
            brier,

        "ece":
            ece,

        **stats,
    }


    final_trades[
        "test_year"
    ] = test_year


    final_trades[
        "calibration"
    ] = calibration_method


    final_trades[
        "threshold"
    ] = threshold


    final_trades[
        "session"
    ] = session


    final_trades[
        "sizing_policy"
    ] = sizing_policy


    return (

        row,

        final_trades,

        {
            "calibration":
                calibration_table,

            "threshold":
                threshold_table,

            "sizing":
                sizing_table,
        }
    )


# ============================================================
# 23. RUN 2020-2026
# ============================================================

annual_rows = []

trade_frames = []

selection_details = {}


print()
print("=" * 100)
print("CLEAN BASE HGB REVALIDATION")
print("=" * 100)


for year in (

    DEVELOPMENT_YEARS

    +

    [
        CONFIRMATION_YEAR
    ]
):

    print()
    print(
        f"TEST YEAR {year}"
    )

    print(
        "-" * 60
    )


    result = evaluate_year(

        DATA,

        year
    )


    if result is None:

        print(
            "SKIPPED"
        )

        continue


    (
        row,
        trades,
        details
    ) = result


    annual_rows.append(
        row
    )


    trade_frames.append(
        trades
    )


    selection_details[
        year
    ] = details


    print(
        "Calibration:",
        row[
            "calibration"
        ]
    )

    print(
        "Threshold:",
        row[
            "threshold"
        ]
    )

    print(
        "Session:",
        row[
            "session"
        ]
    )

    print(
        "Sizing:",
        row[
            "sizing"
        ]
    )

    print(
        f"AUC: {row['auc']:.4f}"
    )

    print(
        "Trades:",
        row[
            "trades"
        ]
    )

    print(
        f"Win: {row['win_rate']*100:.2f}%"
    )

    print(
        f"Avg Return: {row['avg_return']*100:.5f}%"
    )

    print(
        f"PF: {row['profit_factor']:.3f}"
    )

    print(
        f"Growth: {row['growth']*100:.3f}%"
    )

    print(
        f"Max DD: {row['max_dd']*100:.3f}%"
    )

    print(
        f"Return/DD: {row['return_to_dd']:.3f}"
    )


ANNUAL = pd.DataFrame(
    annual_rows
)


TRADES = pd.concat(
    trade_frames
).sort_index()


# ============================================================
# 24. DEVELOPMENT SUMMARY 2020-2025
# ============================================================

DEV_ANNUAL = ANNUAL.loc[

    ANNUAL[
        "test_year"
    ].isin(
        DEVELOPMENT_YEARS
    )

].copy()


DEV_TRADES = TRADES.loc[

    TRADES[
        "test_year"
    ].isin(
        DEVELOPMENT_YEARS
    )

].copy()


DEV_STATS = stats_of_returns(

    DEV_TRADES[
        "net_return"
    ]
)


DEVELOPMENT_SUMMARY = pd.DataFrame(

    [

        {

            "years":
                len(
                    DEV_ANNUAL
                ),

            "positive_years":
                int(
                    (
                        DEV_ANNUAL[
                            "avg_return"
                        ]
                        >
                        0
                    ).sum()
                ),

            "pf_above_1_years":
                int(
                    (
                        DEV_ANNUAL[
                            "profit_factor"
                        ]
                        >
                        1
                    ).sum()
                ),

            "mean_auc":
                DEV_ANNUAL[
                    "auc"
                ].mean(),

            "mean_brier":
                DEV_ANNUAL[
                    "brier"
                ].mean(),

            "mean_ece":
                DEV_ANNUAL[
                    "ece"
                ].mean(),

            **DEV_STATS,
        }
    ]
)


# ============================================================
# 25. 2026 CONFIRMATION
# ============================================================

CONFIRMATION = ANNUAL.loc[

    ANNUAL[
        "test_year"
    ]

    ==

    CONFIRMATION_YEAR

].copy()


# ============================================================
# 26. COST STRESS
# ============================================================

cost_rows = []


for multiplier in [
    1.0,
    1.5,
    2.0,
]:

    returns = (

        DEV_TRADES[
            "position_size"
        ].to_numpy()

        *

        (

            DEV_TRADES[
                "gross_return"
            ].to_numpy()

            -

            BASE_COST
            *
            multiplier
        )
    )


    stats = stats_of_returns(
        returns
    )


    cost_rows.append(

        {

            "cost_x":
                multiplier,

            **stats,
        }
    )


COST_STRESS = pd.DataFrame(
    cost_rows
)


# ============================================================
# 27. 2022 / 2025 OUTLIER AUDIT
# ============================================================

outlier_rows = []


for year in [
    2022,
    2025,
]:

    subset = TRADES.loc[

        TRADES[
            "test_year"
        ]
        ==
        year

    ].copy()


    if len(
        subset
    ) == 0:

        continue


    returns = (

        subset[
            "net_return"
        ]
        .dropna()
        .sort_values(
            ascending=False
        )
    )


    positive = returns.loc[
        returns > 0
    ]


    total_positive = (
        positive.sum()
    )


    def positive_share(
        n
    ):

        if total_positive <= 0:

            return np.nan


        return float(

            positive
            .head(
                n
            )
            .sum()

            /

            total_positive
        )


    stats = stats_of_returns(
        returns
    )


    outlier_rows.append(

        {

            "year":
                year,

            **stats,

            "largest_trade":
                float(
                    returns.iloc[0]
                ),

            "worst_trade":
                float(
                    returns.iloc[-1]
                ),

            "top1_profit_share":
                positive_share(
                    1
                ),

            "top5_profit_share":
                positive_share(
                    5
                ),

            "top10_profit_share":
                positive_share(
                    10
                ),
        }
    )


OUTLIER_AUDIT = pd.DataFrame(
    outlier_rows
)


# ============================================================
# 28. EXTREME TRADE AUDIT
# ============================================================

EXTREME_30M = TRADES.loc[

    TRADES[
        "future_return"
    ].abs()

    >

    0.02

].copy()


# ============================================================
# 29. OVERLAP / DUPLICATE CHECK
# ============================================================

duplicate_signals = int(

    TRADES.index
    .duplicated()
    .sum()
)


overlap_count = 0


for year, group in (
    TRADES.groupby(
        "test_year"
    )
):

    g = group.sort_values(
        "entry_time"
    )


    previous_exit = (

        g[
            "label_end"
        ]
        .shift(1)
    )


    overlap_count += int(

        (
            g[
                "entry_time"
            ]

            <

            previous_exit
        )
        .fillna(
            False
        )
        .sum()
    )


# ============================================================
# 30. AUTOMATIC DECISION
# ============================================================

dev = DEVELOPMENT_SUMMARY.iloc[
    0
]


confirmation_ok = False


if len(
    CONFIRMATION
):

    conf = CONFIRMATION.iloc[
        0
    ]


    confirmation_ok = (

        conf[
            "avg_return"
        ]
        >
        0

        and

        conf[
            "profit_factor"
        ]
        >
        1
    )


cost_2x = COST_STRESS.loc[

    COST_STRESS[
        "cost_x"
    ]

    ==

    2.0

].iloc[0]


cost_ok = (

    cost_2x[
        "avg_return"
    ]
    >
    0

    and

    cost_2x[
        "profit_factor"
    ]
    >
    1
)


development_ok = (

    dev[
        "positive_years"
    ]
    >=
    5

    and

    dev[
        "pf_above_1_years"
    ]
    >=
    5

    and

    dev[
        "avg_return"
    ]
    >
    0

    and

    dev[
        "profit_factor"
    ]
    >
    1
)


integrity_ok = (

    duplicate_signals
    ==
    0

    and

    overlap_count
    ==
    0
)


if (

    development_ok

    and

    confirmation_ok

    and

    cost_ok

    and

    integrity_ok
):

    FINAL_DECISION = (

        "CLEAN_BASE_PIPELINE_PASSES_REVALIDATION"
    )


else:

    FINAL_DECISION = (

        "CLEAN_BASE_PIPELINE_DOES_NOT_YET_PASS"
    )


# ============================================================
# 31. WARNINGS
# ============================================================

WARNINGS = []


high_pf_years = ANNUAL.loc[

    ANNUAL[
        "profit_factor"
    ]
    >
    5
]


if len(
    high_pf_years
):

    WARNINGS.append(

        f"PF > 5 の年が {len(high_pf_years)} 件あります。"
    )


if len(
    OUTLIER_AUDIT
):

    concentration = OUTLIER_AUDIT.loc[

        OUTLIER_AUDIT[
            "top10_profit_share"
        ]

        >
        0.50
    ]


    if len(
        concentration
    ):

        WARNINGS.append(

            "2022/2025で上位10取引が"
            "総利益の50%以上を占める年があります。"
        )


if len(
    EXTREME_30M
):

    WARNINGS.append(

        f"30分future returnが±2%超の取引が"
        f" {len(EXTREME_30M)} 件あります。"
    )


# ============================================================
# 32. DISPLAY
# ============================================================

def show(
    title,
    frame
):

    print()
    print("=" * 100)
    print(title)
    print("=" * 100)

    if frame is None or len(
        frame
    ) == 0:

        print(
            "No data"
        )

        return


    try:

        display(
            frame
        )

    except Exception:

        print(
            frame.to_string(
                index=False
            )
        )


show(
    "ANNUAL OOS RESULTS",
    ANNUAL
)


show(
    "DEVELOPMENT OOS 2020-2025",
    DEVELOPMENT_SUMMARY
)


show(
    "2026 CONFIRMATION",
    CONFIRMATION
)


show(
    "COST STRESS",
    COST_STRESS
)


show(
    "2022 / 2025 OUTLIER AUDIT",
    OUTLIER_AUDIT
)


if len(
    EXTREME_30M
):

    extreme_columns = [

        "test_year",
        "side",
        "confidence",
        "position_size",
        "future_return",
        "gross_return",
        "net_return",
    ]


    show(
        "EXTREME 30M TRADES >2%",
        EXTREME_30M[
            extreme_columns
        ]
    )


# ============================================================
# 33. FINAL DIAGNOSTIC
# ============================================================

print()
print("=" * 100)
print("FINAL CLEAN REVALIDATION DIAGNOSIS")
print("=" * 100)

print(
    "Development positive years:",
    int(
        dev[
            "positive_years"
        ]
    ),
    "/",
    int(
        dev[
            "years"
        ]
    )
)

print(
    "Development PF > 1 years:",
    int(
        dev[
            "pf_above_1_years"
        ]
    ),
    "/",
    int(
        dev[
            "years"
        ]
    )
)

print(
    "Development aggregate PF:",
    round(
        float(
            dev[
                "profit_factor"
            ]
        ),
        4
    )
)

print(
    "Development Avg Return:",
    f"{float(dev['avg_return'])*100:.5f}%"
)

print(
    "2026 confirmation:",
    confirmation_ok
)

print(
    "2x cost survives:",
    cost_ok
)

print(
    "Duplicate signals:",
    duplicate_signals
)

print(
    "Overlapping positions:",
    overlap_count
)

print(
    "Extreme >2% 30m trades:",
    len(
        EXTREME_30M
    )
)

print()
print(
    "FINAL DECISION:",
    FINAL_DECISION
)


if WARNINGS:

    print()
    print(
        "WARNINGS:"
    )

    for warning in WARNINGS:

        print(
            " -",
            warning
        )


if FINAL_DECISION == (
    "CLEAN_BASE_PIPELINE_PASSES_REVALIDATION"
):

    print()
    print(
        "次の段階:"
    )

    print(
        "BASE vs BASE+REGIME vs BASE+VOL+REGIME"
    )

    print(
        "FINAL FEATURE TOURNAMENTへ進む。"
    )

else:

    print()
    print(
        "Feature Tournamentへ進まず、"
        "clean data上でBASEが弱くなった原因を診断する。"
    )


# ============================================================
# 34. SAVE NOTEBOOK VARIABLES
# ============================================================

CLEAN_BASE_ANNUAL = (
    ANNUAL.copy()
)

CLEAN_BASE_TRADES = (
    TRADES.copy()
)

CLEAN_BASE_DEVELOPMENT = (
    DEVELOPMENT_SUMMARY.copy()
)

CLEAN_BASE_CONFIRMATION = (
    CONFIRMATION.copy()
)

CLEAN_BASE_COST_STRESS = (
    COST_STRESS.copy()
)

CLEAN_BASE_OUTLIER_AUDIT = (
    OUTLIER_AUDIT.copy()
)

CLEAN_BASE_EXTREME_TRADES = (
    EXTREME_30M.copy()
)

CLEAN_BASE_SELECTION_DETAILS = (
    selection_details
)

CLEAN_BASE_FINAL_DECISION = (
    FINAL_DECISION
)


print()
print("=" * 100)
print("REVALIDATION COMPLETE")
print("=" * 100)

print(
    "Saved variables:"
)

print(
    "CLEAN_BASE_ANNUAL"
)

print(
    "CLEAN_BASE_TRADES"
)

print(
    "CLEAN_BASE_DEVELOPMENT"
)

print(
    "CLEAN_BASE_CONFIRMATION"
)

print(
    "CLEAN_BASE_COST_STRESS"
)

print(
    "CLEAN_BASE_OUTLIER_AUDIT"
)

print(
    "CLEAN_BASE_FINAL_DECISION"
)


## 元セルindex 67
構文状態：valid


In [ ]:
# ============================================================
# FINAL CLEAN FEATURE TOURNAMENT
#
# BASE
# vs
# BASE + REGIME
# vs
# BASE + VOLATILITY + REGIME
#
# IMPORTANT:
# - CLEAN data only
# - NO fallback model
# - NO fallback feature list
# - Existing CLEAN BASE pipeline logic is reused exactly
# - 2020-2025 = Development OOS
# - 2026      = Confirmation (NOT pristine holdout)
# - Fixed 30m exit
# - Non-overlapping positions
# - Cost = existing CLEAN BASE setting
#
# Goal:
# Choose the simplest feature set that clearly beats BASE.
# ============================================================

import math
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 280)
pd.set_option("display.max_rows", 300)


# ============================================================
# 0. HARD PRECHECK
# ============================================================

REQUIRED_OBJECTS = [
    "bars",
    "BASE_FEATURES",
    "HGB_CONFIG",
    "BASE_COST",
    "DEVELOPMENT_YEARS",
    "CONFIRMATION_YEAR",
    "make_base_features",
    "evaluate_year",
    "stats_of_returns",
    "CLEAN_BASE_ANNUAL",
    "CLEAN_BASE_FINAL_DECISION",
]

missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "前の CLEAN BASE FULL REVALIDATION セルの"
        "実行結果が不足しています。\n"
        f"Missing: {missing_objects}\n\n"
        "前のClean BASE Revalidationセルを先に実行してください。"
    )


if (
    CLEAN_BASE_FINAL_DECISION
    !=
    "CLEAN_BASE_PIPELINE_PASSES_REVALIDATION"
):

    raise RuntimeError(
        "Clean BASE PipelineがRevalidationを通過していません。\n"
        "Final Feature Tournamentへ進めません。"
    )


# ============================================================
# 1. FORMAL BASE 30 FEATURES
# ============================================================

FORMAL_BASE_FEATURES = [

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",

    "weekday",
]


# ============================================================
# 2. FORMAL REGIME 11
#
# Prior reintegration definition
# ============================================================

REGIME_FEATURES = [

    "adx14",
    "adx28",

    "ma20_50_spread",
    "ma50_100_spread",

    "trend_strength_20",
    "trend_strength_50",

    "price_pos_20",
    "price_pos_50",

    "ma_alignment_score",
    "slope_alignment_score",

    "directional_persistence_16",
]


# ============================================================
# 3. FORMAL VOLATILITY 13
#
# Prior reintegration definition
# ============================================================

VOLATILITY_FEATURES = [

    "vol_64",
    "vol_96",

    "vol_ratio_4_32",
    "vol_ratio_8_32",
    "vol_ratio_16_64",

    "range_mean_8",
    "range_mean_32",
    "range_ratio_8_32",

    "range_z_20",
    "range_z_50",

    "abs_return_z_32",
    "abs_return_z_96",

    "gap_abs_1",
]


FEATURE_SETS = {

    "BASE":
        FORMAL_BASE_FEATURES,

    "BASE_PLUS_REGIME":
        FORMAL_BASE_FEATURES
        +
        REGIME_FEATURES,

    "BASE_PLUS_VOL_REGIME":
        FORMAL_BASE_FEATURES
        +
        VOLATILITY_FEATURES
        +
        REGIME_FEATURES,
}


# ============================================================
# 4. BASE LIST MUST MATCH PREVIOUS CLEAN RUN
# ============================================================

if list(BASE_FEATURES) != FORMAL_BASE_FEATURES:

    raise RuntimeError(
        "現在のBASE_FEATURESが正式30特徴量と一致しません。\n"
        "Final Tournamentではfallbackしません。\n"
        f"Current : {list(BASE_FEATURES)}\n"
        f"Expected: {FORMAL_BASE_FEATURES}"
    )


if len(FORMAL_BASE_FEATURES) != 30:

    raise RuntimeError(
        "BASE feature count != 30"
    )


if len(REGIME_FEATURES) != 11:

    raise RuntimeError(
        "REGIME feature count != 11"
    )


if len(VOLATILITY_FEATURES) != 13:

    raise RuntimeError(
        "VOLATILITY feature count != 13"
    )


print("=" * 110)
print("FINAL CLEAN FEATURE TOURNAMENT")
print("=" * 110)

for name, cols in FEATURE_SETS.items():

    print(
        f"{name}: {len(cols)} features"
    )

print()
print(
    "HGB:",
    HGB_CONFIG
)

print(
    "Cost:",
    BASE_COST
)

print(
    "NO FALLBACK MODEL"
)

print(
    "NO FALLBACK FEATURE LIST"
)


# ============================================================
# 5. CLEAN BARS CHECK
# ============================================================

B = bars.copy()

B.columns = [
    str(c).strip().lower()
    for c in B.columns
]


required_ohlc = [
    "open",
    "high",
    "low",
    "close",
]


missing_ohlc = [
    c
    for c in required_ohlc
    if c not in B.columns
]


if missing_ohlc:

    raise RuntimeError(
        f"OHLC missing: {missing_ohlc}"
    )


B = B[
    required_ohlc
].copy()


if not isinstance(
    B.index,
    pd.DatetimeIndex
):

    raise RuntimeError(
        "bars.index must be DatetimeIndex"
    )


B.index = pd.to_datetime(
    B.index,
    utc=True,
    errors="coerce"
)


B = B.loc[
    ~B.index.isna()
].copy()


B = B.sort_index()


if B.index.duplicated().any():

    raise RuntimeError(
        "Duplicate timestamp detected."
    )


bad_grid = (

    (B.index.minute % 15 != 0)

    |

    (B.index.second != 0)

    |

    (B.index.microsecond != 0)
)


if np.asarray(
    bad_grid
).any():

    raise RuntimeError(
        "Non-15m timestamp detected."
    )


for c in required_ohlc:

    B[c] = pd.to_numeric(
        B[c],
        errors="coerce"
    )


if B[
    required_ohlc
].isna().any().any():

    raise RuntimeError(
        "OHLC contains NaN."
    )


print()
print(
    "Clean rows:",
    f"{len(B):,}"
)

print(
    "Period:",
    B.index.min(),
    "->",
    B.index.max()
)


# ============================================================
# 6. EXACT AUXILIARY FUNCTIONS
# ============================================================

def tournament_true_range(x):

    previous_close = (
        x["close"]
        .shift(1)
    )

    return pd.concat(

        [

            x["high"]
            -
            x["low"],

            (
                x["high"]
                -
                previous_close
            ).abs(),

            (
                x["low"]
                -
                previous_close
            ).abs(),
        ],

        axis=1

    ).max(
        axis=1
    )


def tournament_calc_adx(
    x,
    period=14
):

    high = x[
        "high"
    ]

    low = x[
        "low"
    ]


    up_move = (
        high.diff()
    )

    down_move = (
        -low.diff()
    )


    plus_dm = pd.Series(

        np.where(

            (
                up_move
                >
                down_move
            )

            &

            (
                up_move
                >
                0
            ),

            up_move,

            0.0
        ),

        index=x.index
    )


    minus_dm = pd.Series(

        np.where(

            (
                down_move
                >
                up_move
            )

            &

            (
                down_move
                >
                0
            ),

            down_move,

            0.0
        ),

        index=x.index
    )


    tr = tournament_true_range(
        x
    )


    tr_sum = (

        tr
        .rolling(
            period
        )
        .sum()
        .replace(
            0,
            np.nan
        )
    )


    plus_di = (

        100.0

        *

        plus_dm
        .rolling(
            period
        )
        .sum()

        /

        tr_sum
    )


    minus_di = (

        100.0

        *

        minus_dm
        .rolling(
            period
        )
        .sum()

        /

        tr_sum
    )


    denominator = (

        plus_di
        +
        minus_di

    ).replace(
        0,
        np.nan
    )


    dx = (

        100.0

        *

        (
            plus_di
            -
            minus_di
        ).abs()

        /

        denominator
    )


    return (

        dx
        .rolling(
            period
        )
        .mean()

        /

        100.0
    )


def tournament_rolling_z(
    series,
    window
):

    mean = (
        series
        .rolling(
            window
        )
        .mean()
    )


    std = (

        series
        .rolling(
            window
        )
        .std()
        .replace(
            0,
            np.nan
        )
    )


    return (

        series
        -
        mean

    ) / std


# ============================================================
# 7. BUILD ALL FEATURES
#
# BASE part uses the EXACT previous CLEAN function.
# ============================================================

def make_final_tournament_features(
    bars_input
):

    raw = bars_input.copy()


    # --------------------------------------------------------
    # Exact CLEAN BASE features
    # --------------------------------------------------------

    base = make_base_features(
        bars_input
    )


    x = raw.copy()


    for feature in FORMAL_BASE_FEATURES:

        if feature not in base.columns:

            raise RuntimeError(
                f"BASE feature missing: {feature}"
            )

        x[
            feature
        ] = base[
            feature
        ]


    eps = 1e-12


    # ========================================================
    # VOLATILITY 13
    # ========================================================

    x[
        "vol_64"
    ] = (

        x[
            "return_1"
        ]
        .rolling(
            64
        )
        .std()
    )


    x[
        "vol_96"
    ] = (

        x[
            "return_1"
        ]
        .rolling(
            96
        )
        .std()
    )


    x[
        "vol_ratio_4_32"
    ] = (

        x[
            "vol_4"
        ]

        /

        (
            x[
                "vol_32"
            ].abs()

            +
            eps
        )
    )


    x[
        "vol_ratio_8_32"
    ] = (

        x[
            "vol_8"
        ]

        /

        (
            x[
                "vol_32"
            ].abs()

            +
            eps
        )
    )


    x[
        "vol_ratio_16_64"
    ] = (

        x[
            "vol_16"
        ]

        /

        (
            x[
                "vol_64"
            ].abs()

            +
            eps
        )
    )


    x[
        "range_mean_8"
    ] = (

        x[
            "range_pct"
        ]
        .rolling(
            8
        )
        .mean()
    )


    x[
        "range_mean_32"
    ] = (

        x[
            "range_pct"
        ]
        .rolling(
            32
        )
        .mean()
    )


    x[
        "range_ratio_8_32"
    ] = (

        x[
            "range_mean_8"
        ]

        /

        (
            x[
                "range_mean_32"
            ].abs()

            +
            eps
        )
    )


    x[
        "range_z_20"
    ] = tournament_rolling_z(

        x[
            "range_pct"
        ],

        20
    )


    x[
        "range_z_50"
    ] = tournament_rolling_z(

        x[
            "range_pct"
        ],

        50
    )


    abs_return = (

        x[
            "return_1"
        ]
        .abs()
    )


    x[
        "abs_return_z_32"
    ] = tournament_rolling_z(

        abs_return,

        32
    )


    x[
        "abs_return_z_96"
    ] = tournament_rolling_z(

        abs_return,

        96
    )


    x[
        "gap_abs_1"
    ] = (

        x[
            "open"
        ]

        /

        x[
            "close"
        ].shift(
            1
        )

        -

        1

    ).abs()


    # ========================================================
    # REGIME 11
    # ========================================================

    x[
        "adx14"
    ] = tournament_calc_adx(

        x,

        14
    )


    x[
        "adx28"
    ] = tournament_calc_adx(

        x,

        28
    )


    ma20 = (

        x[
            "close"
        ]
        .rolling(
            20
        )
        .mean()
    )


    ma50 = (

        x[
            "close"
        ]
        .rolling(
            50
        )
        .mean()
    )


    ma100 = (

        x[
            "close"
        ]
        .rolling(
            100
        )
        .mean()
    )


    x[
        "ma20_50_spread"
    ] = (

        ma20

        /

        ma50

        -

        1
    )


    x[
        "ma50_100_spread"
    ] = (

        ma50

        /

        ma100

        -

        1
    )


    true_range = tournament_true_range(
        x
    )


    atr14_absolute = (

        true_range
        .rolling(
            14
        )
        .mean()
    )


    atr_safe = (

        atr14_absolute
        .replace(
            0,
            np.nan
        )
    )


    x[
        "trend_strength_20"
    ] = (

        (
            x[
                "close"
            ]
            -
            ma20
        ).abs()

        /

        atr_safe
    )


    x[
        "trend_strength_50"
    ] = (

        (
            x[
                "close"
            ]
            -
            ma50
        ).abs()

        /

        atr_safe
    )


    high20 = (

        x[
            "high"
        ]
        .rolling(
            20
        )
        .max()
    )


    low20 = (

        x[
            "low"
        ]
        .rolling(
            20
        )
        .min()
    )


    high50 = (

        x[
            "high"
        ]
        .rolling(
            50
        )
        .max()
    )


    low50 = (

        x[
            "low"
        ]
        .rolling(
            50
        )
        .min()
    )


    x[
        "price_pos_20"
    ] = (

        (
            x[
                "close"
            ]
            -
            low20
        )

        /

        (
            high20
            -
            low20
        ).replace(
            0,
            np.nan
        )

        -

        0.5
    )


    x[
        "price_pos_50"
    ] = (

        (
            x[
                "close"
            ]
            -
            low50
        )

        /

        (
            high50
            -
            low50
        ).replace(
            0,
            np.nan
        )

        -

        0.5
    )


    x[
        "ma_alignment_score"
    ] = (

        (
            ma20
            >
            ma50
        ).astype(
            float
        )

        +

        (
            ma50
            >
            ma100
        ).astype(
            float
        )

    ) / 2.0


    slope20 = (
        ma20
        .pct_change()
    )

    slope50 = (
        ma50
        .pct_change()
    )

    slope100 = (
        ma100
        .pct_change()
    )


    x[
        "slope_alignment_score"
    ] = (

        np.sign(
            slope20
        )

        +

        np.sign(
            slope50
        )

        +

        np.sign(
            slope100
        )

    ) / 3.0


    sign_return = np.sign(

        x[
            "return_1"
        ]
    )


    x[
        "directional_persistence_16"
    ] = (

        sign_return

        .rolling(
            16
        )

        .mean()

        .abs()
    )


    return x.replace(

        [
            np.inf,
            -np.inf
        ],

        np.nan
    )


ALL_FEATURE_FRAME = (
    make_final_tournament_features(
        B
    )
)


# ============================================================
# 8. VERIFY FEATURE EXISTENCE
# ============================================================

all_required_features = list(
    dict.fromkeys(

        FORMAL_BASE_FEATURES

        +

        VOLATILITY_FEATURES

        +

        REGIME_FEATURES
    )
)


missing_features = [

    c
    for c in all_required_features

    if c not in ALL_FEATURE_FRAME.columns
]


if missing_features:

    raise RuntimeError(
        f"Feature construction failed: {missing_features}"
    )


print()
print("=" * 110)
print("FEATURE CONSTRUCTION")
print("=" * 110)

print(
    "BASE:",
    len(
        FORMAL_BASE_FEATURES
    )
)

print(
    "REGIME additions:",
    len(
        REGIME_FEATURES
    )
)

print(
    "VOLATILITY additions:",
    len(
        VOLATILITY_FEATURES
    )
)


# ============================================================
# 9. BUILD CANDIDATE DATASET
#
# signal t
# entry Open(t+1)
# exit Close(t+2)
# ============================================================

def build_tournament_dataset(
    feature_list
):

    x = (
        ALL_FEATURE_FRAME
        .copy()
    )


    times = pd.Series(
        B.index,
        index=B.index
    )


    x[
        "entry_time"
    ] = times.shift(
        -1
    )


    x[
        "exit_bar_time"
    ] = times.shift(
        -2
    )


    x[
        "label_end"
    ] = (

        x[
            "exit_bar_time"
        ]

        +

        pd.Timedelta(
            minutes=15
        )
    )


    x[
        "entry_price"
    ] = (

        B[
            "open"
        ]
        .shift(
            -1
        )
    )


    x[
        "exit_price"
    ] = (

        B[
            "close"
        ]
        .shift(
            -2
        )
    )


    x[
        "future_return"
    ] = (

        x[
            "exit_price"
        ]

        /

        x[
            "entry_price"
        ]

        -

        1
    )


    x[
        "target"
    ] = (

        x[
            "future_return"
        ]

        >
        0

    ).astype(
        int
    )


    continuous = (

        (
            x[
                "entry_time"
            ]

            -
            times
        )

        ==

        pd.Timedelta(
            minutes=15
        )

    ) & (

        (
            x[
                "exit_bar_time"
            ]

            -
            times
        )

        ==

        pd.Timedelta(
            minutes=30
        )
    )


    required_columns = (

        list(
            feature_list
        )

        +

        [
            "entry_time",
            "exit_bar_time",
            "label_end",
            "entry_price",
            "exit_price",
            "future_return",
            "target",
        ]
    )


    result = (

        x.loc[
            continuous
        ]

        .dropna(
            subset=required_columns
        )

        .copy()
    )


    return result


# ============================================================
# 10. RUN THE THREE PIPELINES
#
# evaluate_year() is reused from the CLEAN BASE cell.
#
# That means:
# Calibration
# Threshold
# Session
# Sizing
# Exit
# Cost
# are exactly the same.
# ============================================================

ORIGINAL_BASE_FEATURES = list(
    BASE_FEATURES
)


TOURNAMENT_ANNUAL_ROWS = []

TOURNAMENT_TRADE_FRAMES = []

TOURNAMENT_DETAILS = {}

TOURNAMENT_DATA_ROWS = []


try:

    for feature_set_name, feature_list in (
        FEATURE_SETS.items()
    ):

        print()
        print("=" * 110)
        print(
            "RUNNING:",
            feature_set_name
        )
        print("=" * 110)


        # ---------------------------------------------
        # Critical:
        # existing pipeline functions read BASE_FEATURES
        # ---------------------------------------------

        globals()[
            "BASE_FEATURES"
        ] = list(
            feature_list
        )


        candidate_data = (
            build_tournament_dataset(
                feature_list
            )
        )


        TOURNAMENT_DATA_ROWS.append(

            {
                "feature_set":
                    feature_set_name,

                "features":
                    len(
                        feature_list
                    ),

                "usable_rows":
                    len(
                        candidate_data
                    ),
            }
        )


        print(
            "Features:",
            len(
                feature_list
            )
        )

        print(
            "Usable rows:",
            f"{len(candidate_data):,}"
        )


        TOURNAMENT_DETAILS[
            feature_set_name
        ] = {}


        for year in (

            list(
                DEVELOPMENT_YEARS
            )

            +

            [
                CONFIRMATION_YEAR
            ]
        ):

            print(
                f"  Test {year} ...",
                end=" "
            )


            result = evaluate_year(

                candidate_data,

                year
            )


            if result is None:

                raise RuntimeError(
                    f"{feature_set_name} / {year}: "
                    "evaluate_year returned None."
                )


            (
                row,
                trades,
                details
            ) = result


            row = dict(
                row
            )


            row[
                "feature_set"
            ] = feature_set_name


            row[
                "feature_count"
            ] = len(
                feature_list
            )


            trades = trades.copy()


            trades[
                "feature_set"
            ] = feature_set_name


            TOURNAMENT_ANNUAL_ROWS.append(
                row
            )


            TOURNAMENT_TRADE_FRAMES.append(
                trades
            )


            TOURNAMENT_DETAILS[
                feature_set_name
            ][
                year
            ] = details


            print(
                f"PF={row['profit_factor']:.3f}, "
                f"Avg={row['avg_return']*100:.5f}%, "
                f"Trades={row['trades']}"
            )


finally:

    # Never leave notebook BASE_FEATURES modified
    globals()[
        "BASE_FEATURES"
    ] = ORIGINAL_BASE_FEATURES


TOURNAMENT_ANNUAL = pd.DataFrame(
    TOURNAMENT_ANNUAL_ROWS
)


TOURNAMENT_TRADES = pd.concat(

    TOURNAMENT_TRADE_FRAMES,

    axis=0

).sort_index()


TOURNAMENT_DATA_INFO = pd.DataFrame(
    TOURNAMENT_DATA_ROWS
)


# ============================================================
# 11. BASE REPRODUCTION CHECK
#
# Mandatory.
# If this fails -> STOP.
# ============================================================

NEW_BASE = (

    TOURNAMENT_ANNUAL.loc[

        TOURNAMENT_ANNUAL[
            "feature_set"
        ]

        ==

        "BASE"

    ]

    .copy()
)


OLD_BASE = (
    CLEAN_BASE_ANNUAL
    .copy()
)


REPRO = NEW_BASE.merge(

    OLD_BASE,

    on="test_year",

    suffixes=(
        "_new",
        "_old"
    )
)


repro_rows = []


for row in REPRO.itertuples():

    repro_rows.append(

        {
            "test_year":
                row.test_year,

            "trades_equal":
                int(
                    row.trades_new
                )
                ==
                int(
                    row.trades_old
                ),

            "auc_diff":
                abs(
                    row.auc_new
                    -
                    row.auc_old
                ),

            "avg_return_diff":
                abs(
                    row.avg_return_new
                    -
                    row.avg_return_old
                ),

            "pf_diff":
                abs(
                    row.profit_factor_new
                    -
                    row.profit_factor_old
                ),
        }
    )


BASE_REPRODUCTION = pd.DataFrame(
    repro_rows
)


BASE_REPRODUCTION_OK = (

    len(
        BASE_REPRODUCTION
    )
    ==
    len(
        list(
            DEVELOPMENT_YEARS
        )
        +
        [
            CONFIRMATION_YEAR
        ]
    )

    and

    BASE_REPRODUCTION[
        "trades_equal"
    ].all()

    and

    (
        BASE_REPRODUCTION[
            "auc_diff"
        ]
        <
        1e-10
    ).all()

    and

    (
        BASE_REPRODUCTION[
            "avg_return_diff"
        ]
        <
        1e-12
    ).all()

    and

    (
        BASE_REPRODUCTION[
            "pf_diff"
        ]
        <
        1e-9
    ).all()
)


# ============================================================
# 12. DEVELOPMENT SUMMARY
# ============================================================

development_rows = []


for feature_set_name in FEATURE_SETS:

    annual = (

        TOURNAMENT_ANNUAL.loc[

            (
                TOURNAMENT_ANNUAL[
                    "feature_set"
                ]
                ==
                feature_set_name
            )

            &

            (
                TOURNAMENT_ANNUAL[
                    "test_year"
                ]
                .isin(
                    DEVELOPMENT_YEARS
                )
            )

        ]

        .copy()
    )


    trades = (

        TOURNAMENT_TRADES.loc[

            (
                TOURNAMENT_TRADES[
                    "feature_set"
                ]
                ==
                feature_set_name
            )

            &

            (
                TOURNAMENT_TRADES[
                    "test_year"
                ]
                .isin(
                    DEVELOPMENT_YEARS
                )
            )

        ]

        .sort_index()
        .copy()
    )


    stats = stats_of_returns(

        trades[
            "net_return"
        ]
    )


    development_rows.append(

        {
            "feature_set":
                feature_set_name,

            "feature_count":
                len(
                    FEATURE_SETS[
                        feature_set_name
                    ]
                ),

            "years":
                len(
                    annual
                ),

            "positive_years":
                int(
                    (
                        annual[
                            "avg_return"
                        ]
                        >
                        0
                    ).sum()
                ),

            "pf_above_1_years":
                int(
                    (
                        annual[
                            "profit_factor"
                        ]
                        >
                        1
                    ).sum()
                ),

            "mean_auc":
                float(
                    annual[
                        "auc"
                    ].mean()
                ),

            **stats,
        }
    )


DEVELOPMENT_SUMMARY = pd.DataFrame(
    development_rows
)


# ============================================================
# 13. COST STRESS
# ============================================================

cost_rows = []


for feature_set_name in FEATURE_SETS:

    dev = (

        TOURNAMENT_TRADES.loc[

            (
                TOURNAMENT_TRADES[
                    "feature_set"
                ]
                ==
                feature_set_name
            )

            &

            (
                TOURNAMENT_TRADES[
                    "test_year"
                ]
                .isin(
                    DEVELOPMENT_YEARS
                )
            )

        ]

        .sort_index()
        .copy()
    )


    for multiplier in [
        1.0,
        1.5,
        2.0,
    ]:

        stressed_returns = (

            dev[
                "position_size"
            ].to_numpy()

            *

            (

                dev[
                    "gross_return"
                ].to_numpy()

                -

                BASE_COST
                *
                multiplier
            )
        )


        stats = stats_of_returns(
            stressed_returns
        )


        cost_rows.append(

            {
                "feature_set":
                    feature_set_name,

                "cost_x":
                    multiplier,

                **stats,
            }
        )


COST_STRESS = pd.DataFrame(
    cost_rows
)


# ============================================================
# 14. 2026 CONFIRMATION
# ============================================================

CONFIRMATION_TABLE = (

    TOURNAMENT_ANNUAL.loc[

        TOURNAMENT_ANNUAL[
            "test_year"
        ]

        ==

        CONFIRMATION_YEAR

    ]

    .copy()
)


# ============================================================
# 15. LEAVE-ONE-YEAR-OUT ROBUSTNESS
#
# Not retraining.
# Measures dependence of aggregate result on one year.
# ============================================================

loo_rows = []


for feature_set_name in FEATURE_SETS:

    for left_out_year in (
        DEVELOPMENT_YEARS
    ):

        subset = (

            TOURNAMENT_TRADES.loc[

                (
                    TOURNAMENT_TRADES[
                        "feature_set"
                    ]
                    ==
                    feature_set_name
                )

                &

                (
                    TOURNAMENT_TRADES[
                        "test_year"
                    ]
                    .isin(
                        DEVELOPMENT_YEARS
                    )
                )

                &

                (
                    TOURNAMENT_TRADES[
                        "test_year"
                    ]
                    !=
                    left_out_year
                )

            ]

            .sort_index()
        )


        stats = stats_of_returns(

            subset[
                "net_return"
            ]
        )


        loo_rows.append(

            {
                "feature_set":
                    feature_set_name,

                "left_out_year":
                    left_out_year,

                **stats,
            }
        )


LEAVE_ONE_YEAR_OUT = pd.DataFrame(
    loo_rows
)


LOO_SUMMARY = (

    LEAVE_ONE_YEAR_OUT

    .groupby(
        "feature_set",
        as_index=False
    )

    .agg(

        loo_min_pf=(
            "profit_factor",
            "min"
        ),

        loo_mean_pf=(
            "profit_factor",
            "mean"
        ),

        loo_min_avg_return=(
            "avg_return",
            "min"
        ),

        loo_min_return_to_dd=(
            "return_to_dd",
            "min"
        ),
    )
)


# ============================================================
# 16. 2022 LARGE-WINNER SENSITIVITY
#
# Diagnostic ONLY.
# We do NOT delete the trade from the strategy.
# ============================================================

sensitivity_rows = []


for feature_set_name in FEATURE_SETS:

    dev = (

        TOURNAMENT_TRADES.loc[

            (
                TOURNAMENT_TRADES[
                    "feature_set"
                ]
                ==
                feature_set_name
            )

            &

            (
                TOURNAMENT_TRADES[
                    "test_year"
                ]
                .isin(
                    DEVELOPMENT_YEARS
                )
            )

        ]

        .sort_index()
        .copy()
    )


    y2022 = (

        dev.loc[

            dev[
                "test_year"
            ]
            ==
            2022

        ]

        .copy()
    )


    if len(
        y2022
    ) == 0:

        continue


    winning = y2022.loc[

        y2022[
            "net_return"
        ]
        >
        0

    ]


    if len(
        winning
    ) == 0:

        continue


    largest_timestamp = (

        winning[
            "net_return"
        ]
        .idxmax()
    )


    largest_trade = winning.loc[
        largest_timestamp
    ]


    # idx may theoretically produce DataFrame
    if isinstance(
        largest_trade,
        pd.DataFrame
    ):

        largest_trade = (
            largest_trade
            .iloc[0]
        )


    y2022_without = y2022.drop(
        index=largest_timestamp
    )


    dev_without = dev.loc[
        ~(
            (
                dev.index
                ==
                largest_timestamp
            )

            &

            (
                dev[
                    "test_year"
                ]
                ==
                2022
            )
        )
    ]


    year_stats = stats_of_returns(

        y2022_without[
            "net_return"
        ]
    )


    dev_stats = stats_of_returns(

        dev_without[
            "net_return"
        ]
    )


    sensitivity_rows.append(

        {
            "feature_set":
                feature_set_name,

            "removed_timestamp":
                largest_timestamp,

            "removed_net_return":
                float(
                    largest_trade[
                        "net_return"
                    ]
                ),

            "removed_future_return":
                float(
                    largest_trade[
                        "future_return"
                    ]
                ),

            "2022_pf_without_largest":
                year_stats[
                    "profit_factor"
                ],

            "2022_avg_without_largest":
                year_stats[
                    "avg_return"
                ],

            "dev_pf_without_2022_largest":
                dev_stats[
                    "profit_factor"
                ],

            "dev_avg_without_2022_largest":
                dev_stats[
                    "avg_return"
                ],

            "dev_rdd_without_2022_largest":
                dev_stats[
                    "return_to_dd"
                ],
        }
    )


SENSITIVITY_2022 = pd.DataFrame(
    sensitivity_rows
)


# ============================================================
# 17. PROFIT CONCENTRATION
# ============================================================

concentration_rows = []


for feature_set_name in FEATURE_SETS:

    for year in [
        2022,
        2025,
    ]:

        subset = (

            TOURNAMENT_TRADES.loc[

                (
                    TOURNAMENT_TRADES[
                        "feature_set"
                    ]
                    ==
                    feature_set_name
                )

                &

                (
                    TOURNAMENT_TRADES[
                        "test_year"
                    ]
                    ==
                    year
                )

            ]

            .copy()
        )


        if len(
            subset
        ) == 0:

            continue


        positive = (

            subset.loc[

                subset[
                    "net_return"
                ]
                >
                0,

                "net_return"

            ]

            .sort_values(
                ascending=False
            )
        )


        total_positive = (
            positive.sum()
        )


        def share(n):

            if total_positive <= 0:

                return np.nan

            return float(

                positive.head(
                    n
                ).sum()

                /

                total_positive
            )


        concentration_rows.append(

            {
                "feature_set":
                    feature_set_name,

                "year":
                    year,

                "trades":
                    len(
                        subset
                    ),

                "top1_profit_share":
                    share(
                        1
                    ),

                "top5_profit_share":
                    share(
                        5
                    ),

                "top10_profit_share":
                    share(
                        10
                    ),

                "largest_net_trade":
                    float(
                        subset[
                            "net_return"
                        ].max()
                    ),

                "worst_net_trade":
                    float(
                        subset[
                            "net_return"
                        ].min()
                    ),
            }
        )


PROFIT_CONCENTRATION = pd.DataFrame(
    concentration_rows
)


# ============================================================
# 18. DUPLICATE / OVERLAP INTEGRITY
# ============================================================

integrity_rows = []


for feature_set_name in FEATURE_SETS:

    trades = (

        TOURNAMENT_TRADES.loc[

            TOURNAMENT_TRADES[
                "feature_set"
            ]

            ==

            feature_set_name

        ]

        .copy()
    )


    duplicate_count = int(

        trades.index
        .duplicated()
        .sum()
    )


    overlap_count = 0


    for year, group in trades.groupby(
        "test_year"
    ):

        g = group.sort_values(
            "entry_time"
        )


        previous_exit = (

            g[
                "label_end"
            ]
            .shift(1)
        )


        overlap_count += int(

            (
                g[
                    "entry_time"
                ]

                <
                previous_exit
            )

            .fillna(
                False
            )

            .sum()
        )


    integrity_rows.append(

        {
            "feature_set":
                feature_set_name,

            "duplicate_signals":
                duplicate_count,

            "overlapping_positions":
                overlap_count,

            "integrity_ok":
                (
                    duplicate_count
                    ==
                    0

                    and

                    overlap_count
                    ==
                    0
                ),
        }
    )


INTEGRITY_TABLE = pd.DataFrame(
    integrity_rows
)


# ============================================================
# 19. PAIRED MOVING-BLOCK BOOTSTRAP
#
# Development only.
# Challenger daily return - BASE daily return.
# ============================================================

def daily_compounded_return(
    trades
):

    if len(
        trades
    ) == 0:

        return pd.Series(
            dtype=float
        )


    temp = trades.copy()


    temp[
        "trade_date"
    ] = (
        temp.index
        .normalize()
    )


    daily = (

        temp
        .groupby(
            "trade_date"
        )[
            "net_return"
        ]

        .apply(

            lambda r:

            float(
                np.prod(
                    1
                    +
                    np.asarray(
                        r,
                        dtype=float
                    )
                )
                -
                1
            )
        )
    )


    return daily.sort_index()


def moving_block_bootstrap_alpha(
    challenger_trades,
    base_trades,
    iterations=2000,
    block_days=10,
    seed=42
):

    challenger_daily = daily_compounded_return(
        challenger_trades
    )


    base_daily = daily_compounded_return(
        base_trades
    )


    dates = (

        challenger_daily.index
        .union(
            base_daily.index
        )
        .sort_values()
    )


    challenger = (

        challenger_daily
        .reindex(
            dates,
            fill_value=0.0
        )
        .to_numpy()
    )


    base = (

        base_daily
        .reindex(
            dates,
            fill_value=0.0
        )
        .to_numpy()
    )


    difference = (

        challenger
        -
        base
    )


    n = len(
        difference
    )


    if n < 20:

        return {

            "observed_daily_alpha":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "p_alpha_gt_0":
                np.nan,
        }


    block = min(
        block_days,
        n
    )


    possible_starts = (

        n
        -
        block
        +
        1
    )


    rng = np.random.default_rng(
        seed
    )


    bootstrap_means = np.empty(
        iterations
    )


    blocks_needed = math.ceil(
        n
        /
        block
    )


    for i in range(
        iterations
    ):

        starts = rng.integers(

            0,

            possible_starts,

            size=blocks_needed
        )


        sample_parts = [

            difference[
                start:
                start
                +
                block
            ]

            for start in starts
        ]


        sample = np.concatenate(
            sample_parts
        )[
            :n
        ]


        bootstrap_means[
            i
        ] = np.mean(
            sample
        )


    return {

        "observed_daily_alpha":
            float(
                difference.mean()
            ),

        "ci_low":
            float(
                np.quantile(
                    bootstrap_means,
                    0.025
                )
            ),

        "ci_high":
            float(
                np.quantile(
                    bootstrap_means,
                    0.975
                )
            ),

        "p_alpha_gt_0":
            float(
                (
                    bootstrap_means
                    >
                    0
                ).mean()
            ),
    }


bootstrap_rows = []


BASE_DEV_TRADES = (

    TOURNAMENT_TRADES.loc[

        (
            TOURNAMENT_TRADES[
                "feature_set"
            ]
            ==
            "BASE"
        )

        &

        (
            TOURNAMENT_TRADES[
                "test_year"
            ]
            .isin(
                DEVELOPMENT_YEARS
            )
        )

    ]

    .copy()
)


for challenger in [

    "BASE_PLUS_REGIME",
    "BASE_PLUS_VOL_REGIME",
]:

    challenger_trades = (

        TOURNAMENT_TRADES.loc[

            (
                TOURNAMENT_TRADES[
                    "feature_set"
                ]
                ==
                challenger
            )

            &

            (
                TOURNAMENT_TRADES[
                    "test_year"
                ]
                .isin(
                    DEVELOPMENT_YEARS
                )
            )

        ]

        .copy()
    )


    result = moving_block_bootstrap_alpha(

        challenger_trades,

        BASE_DEV_TRADES,

        iterations=2000,

        block_days=10,

        seed=42
    )


    bootstrap_rows.append(

        {
            "challenger":
                challenger,

            **result,
        }
    )


PAIRED_BOOTSTRAP = pd.DataFrame(
    bootstrap_rows
)


# ============================================================
# 20. BUILD FINAL COMPARISON TABLE
# ============================================================

cost2 = (

    COST_STRESS.loc[

        COST_STRESS[
            "cost_x"
        ]
        ==
        2.0

    ][
        [
            "feature_set",
            "avg_return",
            "profit_factor",
            "return_to_dd",
        ]
    ]

    .rename(

        columns={

            "avg_return":
                "cost2_avg_return",

            "profit_factor":
                "cost2_pf",

            "return_to_dd":
                "cost2_rdd",
        }
    )
)


confirmation = (

    CONFIRMATION_TABLE[
        [
            "feature_set",
            "avg_return",
            "profit_factor",
            "return_to_dd",
            "trades",
            "auc",
        ]
    ]

    .rename(

        columns={

            "avg_return":
                "confirmation_avg",

            "profit_factor":
                "confirmation_pf",

            "return_to_dd":
                "confirmation_rdd",

            "trades":
                "confirmation_trades",

            "auc":
                "confirmation_auc",
        }
    )
)


sensitivity = (

    SENSITIVITY_2022[
        [
            "feature_set",
            "2022_pf_without_largest",
            "dev_pf_without_2022_largest",
            "dev_avg_without_2022_largest",
        ]
    ]

    .copy()
)


FINAL_COMPARISON = (

    DEVELOPMENT_SUMMARY

    .merge(
        cost2,
        on="feature_set",
        how="left"
    )

    .merge(
        confirmation,
        on="feature_set",
        how="left"
    )

    .merge(
        LOO_SUMMARY,
        on="feature_set",
        how="left"
    )

    .merge(
        sensitivity,
        on="feature_set",
        how="left"
    )

    .merge(
        INTEGRITY_TABLE,
        on="feature_set",
        how="left"
    )
)


# ============================================================
# 21. PREDECLARED CHALLENGER RULE
#
# Development wins:
#   Avg Return
#   PF
#   Return/DD
#   2x Cost PF
#   LOO minimum PF
#
# Need >= 4 / 5
#
# Confirmation wins:
#   Avg Return
#   PF
#   Return/DD
#
# Need >= 2 / 3
#
# Also:
#   Annual stability must not decline
#   2x cost remains profitable
#   2022 top-trade sensitivity remains PF > 1
#   Integrity must pass
# ============================================================

base_row = (

    FINAL_COMPARISON.loc[

        FINAL_COMPARISON[
            "feature_set"
        ]
        ==
        "BASE"

    ]

    .iloc[0]
)


qualification_rows = []


for candidate_name in [

    "BASE_PLUS_REGIME",
    "BASE_PLUS_VOL_REGIME",
]:

    candidate = (

        FINAL_COMPARISON.loc[

            FINAL_COMPARISON[
                "feature_set"
            ]
            ==
            candidate_name

        ]

        .iloc[0]
    )


    dev_comparisons = {

        "dev_avg":
            candidate[
                "avg_return"
            ]
            >
            base_row[
                "avg_return"
            ],

        "dev_pf":
            candidate[
                "profit_factor"
            ]
            >
            base_row[
                "profit_factor"
            ],

        "dev_rdd":
            candidate[
                "return_to_dd"
            ]
            >
            base_row[
                "return_to_dd"
            ],

        "cost2_pf":
            candidate[
                "cost2_pf"
            ]
            >
            base_row[
                "cost2_pf"
            ],

        "loo_min_pf":
            candidate[
                "loo_min_pf"
            ]
            >
            base_row[
                "loo_min_pf"
            ],
    }


    confirmation_comparisons = {

        "confirmation_avg":
            candidate[
                "confirmation_avg"
            ]
            >
            base_row[
                "confirmation_avg"
            ],

        "confirmation_pf":
            candidate[
                "confirmation_pf"
            ]
            >
            base_row[
                "confirmation_pf"
            ],

        "confirmation_rdd":
            candidate[
                "confirmation_rdd"
            ]
            >
            base_row[
                "confirmation_rdd"
            ],
    }


    dev_wins = int(
        sum(
            dev_comparisons.values()
        )
    )


    confirmation_wins = int(
        sum(
            confirmation_comparisons.values()
        )
    )


    stability_ok = (

        candidate[
            "positive_years"
        ]
        >=
        base_row[
            "positive_years"
        ]

        and

        candidate[
            "pf_above_1_years"
        ]
        >=
        base_row[
            "pf_above_1_years"
        ]
    )


    cost_survives = (

        candidate[
            "cost2_pf"
        ]
        >
        1

        and

        candidate[
            "cost2_avg_return"
        ]
        >
        0
    )


    sensitivity_ok = (

        candidate[
            "2022_pf_without_largest"
        ]
        >
        1

        and

        candidate[
            "dev_pf_without_2022_largest"
        ]
        >
        1
    )


    integrity_ok = bool(
        candidate[
            "integrity_ok"
        ]
    )


    qualifies = (

        dev_wins
        >=
        4

        and

        confirmation_wins
        >=
        2

        and

        stability_ok

        and

        cost_survives

        and

        sensitivity_ok

        and

        integrity_ok
    )


    qualification_rows.append(

        {
            "candidate":
                candidate_name,

            "development_wins":
                dev_wins,

            "development_needed":
                4,

            "confirmation_wins":
                confirmation_wins,

            "confirmation_needed":
                2,

            "stability_ok":
                stability_ok,

            "2x_cost_survives":
                cost_survives,

            "2022_sensitivity_ok":
                sensitivity_ok,

            "integrity_ok":
                integrity_ok,

            "qualifies_to_replace_BASE":
                qualifies,

            **{
                f"win_{k}":
                    v

                for k, v in
                dev_comparisons.items()
            },

            **{
                f"win_{k}":
                    v

                for k, v in
                confirmation_comparisons.items()
            },
        }
    )


QUALIFICATION_TABLE = pd.DataFrame(
    qualification_rows
)


# ============================================================
# 22. FINAL CHAMPION DECISION
# ============================================================

if not BASE_REPRODUCTION_OK:

    RECOMMENDED_CHAMPION = None

    FINAL_DECISION = (
        "STOP_BASE_REPRODUCTION_FAILED"
    )


else:

    qualified = (

        QUALIFICATION_TABLE.loc[

            QUALIFICATION_TABLE[
                "qualifies_to_replace_BASE"
            ]

            ==
            True

        ]

        .copy()
    )


    if len(
        qualified
    ) == 0:

        RECOMMENDED_CHAMPION = (
            "BASE"
        )

        FINAL_DECISION = (
            "FREEZE_BASE"
        )


    else:

        # ---------------------------------------------
        # Add bootstrap result as tie-break information
        # ---------------------------------------------

        qualified = qualified.merge(

            PAIRED_BOOTSTRAP[
                [
                    "challenger",
                    "p_alpha_gt_0",
                ]
            ],

            left_on="candidate",

            right_on="challenger",

            how="left"
        )


        qualified[
            "total_metric_wins"
        ] = (

            qualified[
                "development_wins"
            ]

            +

            qualified[
                "confirmation_wins"
            ]
        )


        feature_count_map = {

            name:
                len(
                    features
                )

            for name, features
            in FEATURE_SETS.items()
        }


        qualified[
            "feature_count"
        ] = qualified[
            "candidate"
        ].map(
            feature_count_map
        )


        # Priority:
        # 1. more predefined metric wins
        # 2. stronger paired bootstrap support
        # 3. fewer features

        qualified = qualified.sort_values(

            [
                "total_metric_wins",
                "p_alpha_gt_0",
                "feature_count",
            ],

            ascending=[
                False,
                False,
                True,
            ]
        )


        RECOMMENDED_CHAMPION = str(

            qualified.iloc[0][
                "candidate"
            ]
        )


        FINAL_DECISION = (

            "FREEZE_"
            +
            RECOMMENDED_CHAMPION
        )


# ============================================================
# 23. DISPLAY HELPER
# ============================================================

def tournament_show(
    title,
    frame
):

    print()
    print("=" * 110)
    print(title)
    print("=" * 110)

    if frame is None or len(
        frame
    ) == 0:

        print(
            "No data"
        )

        return


    try:

        display(
            frame
        )

    except Exception:

        print(
            frame.to_string(
                index=False
            )
        )


# ============================================================
# 24. OUTPUT
# ============================================================

tournament_show(
    "FEATURE SET / DATA INFO",
    TOURNAMENT_DATA_INFO
)


tournament_show(
    "BASE REPRODUCTION CHECK",
    BASE_REPRODUCTION
)


print()
print(
    "BASE reproduction passed:",
    BASE_REPRODUCTION_OK
)


tournament_show(
    "ANNUAL OOS RESULTS",
    TOURNAMENT_ANNUAL[
        [
            "feature_set",
            "feature_count",
            "test_year",
            "validation_year",
            "calibration",
            "threshold",
            "session",
            "sizing",
            "auc",
            "trades",
            "win_rate",
            "avg_return",
            "profit_factor",
            "growth",
            "max_dd",
            "return_to_dd",
        ]
    ]
)


tournament_show(
    "DEVELOPMENT 2020-2025",
    DEVELOPMENT_SUMMARY
)


tournament_show(
    "2026 CONFIRMATION",
    CONFIRMATION_TABLE
)


tournament_show(
    "COST STRESS",
    COST_STRESS
)


tournament_show(
    "LEAVE-ONE-YEAR-OUT",
    LEAVE_ONE_YEAR_OUT
)


tournament_show(
    "LOO SUMMARY",
    LOO_SUMMARY
)


tournament_show(
    "2022 LARGE-TRADE SENSITIVITY",
    SENSITIVITY_2022
)


tournament_show(
    "2022 / 2025 PROFIT CONCENTRATION",
    PROFIT_CONCENTRATION
)


tournament_show(
    "INTEGRITY",
    INTEGRITY_TABLE
)


tournament_show(
    "PAIRED MOVING-BLOCK BOOTSTRAP vs BASE",
    PAIRED_BOOTSTRAP
)


tournament_show(
    "FINAL COMPARISON",
    FINAL_COMPARISON
)


tournament_show(
    "CHALLENGER QUALIFICATION",
    QUALIFICATION_TABLE
)


# ============================================================
# 25. FINAL DIAGNOSIS
# ============================================================

print()
print("=" * 110)
print("FINAL FEATURE TOURNAMENT DECISION")
print("=" * 110)

print(
    "BASE reproduction:",
    BASE_REPRODUCTION_OK
)

print(
    "Recommended Champion:",
    RECOMMENDED_CHAMPION
)

print(
    "Decision:",
    FINAL_DECISION
)


if FINAL_DECISION == (
    "FREEZE_BASE"
):

    print()
    print(
        "No challenger cleared the predefined replacement rule."
    )

    print(
        "BASE remains Champion by robustness + simplicity."
    )


elif (
    RECOMMENDED_CHAMPION
    is not None
):

    print()
    print(
        RECOMMENDED_CHAMPION,
        "cleared the predefined rule and can replace BASE."
    )


else:

    print()
    print(
        "STOP: BASE reproduction failed."
    )

    print(
        "Do not freeze a Champion until the discrepancy is resolved."
    )


print()
print(
    "IMPORTANT:"
)

print(
    "2026 is confirmation data already inspected during research,"
)

print(
    "not a globally pristine final holdout."
)

print(
    "After Champion Freeze, the real next validation is forward"
)

print(
    "Shadow / Paper Trading on unseen future data."
)


# ============================================================
# 26. SAVE NOTEBOOK VARIABLES
# ============================================================

FINAL_TOURNAMENT_ANNUAL = (
    TOURNAMENT_ANNUAL.copy()
)

FINAL_TOURNAMENT_TRADES = (
    TOURNAMENT_TRADES.copy()
)

FINAL_TOURNAMENT_DEVELOPMENT = (
    DEVELOPMENT_SUMMARY.copy()
)

FINAL_TOURNAMENT_CONFIRMATION = (
    CONFIRMATION_TABLE.copy()
)

FINAL_TOURNAMENT_COST = (
    COST_STRESS.copy()
)

FINAL_TOURNAMENT_LOO = (
    LEAVE_ONE_YEAR_OUT.copy()
)

FINAL_TOURNAMENT_SENSITIVITY = (
    SENSITIVITY_2022.copy()
)

FINAL_TOURNAMENT_CONCENTRATION = (
    PROFIT_CONCENTRATION.copy()
)

FINAL_TOURNAMENT_BOOTSTRAP = (
    PAIRED_BOOTSTRAP.copy()
)

FINAL_TOURNAMENT_COMPARISON = (
    FINAL_COMPARISON.copy()
)

FINAL_TOURNAMENT_QUALIFICATION = (
    QUALIFICATION_TABLE.copy()
)

FINAL_TOURNAMENT_BASE_REPRODUCTION = (
    BASE_REPRODUCTION.copy()
)

FINAL_TOURNAMENT_RECOMMENDED_CHAMPION = (
    RECOMMENDED_CHAMPION
)

FINAL_TOURNAMENT_DECISION = (
    FINAL_DECISION
)

FINAL_TOURNAMENT_FEATURE_SETS = {

    key:
        list(
            value
        )

    for key, value
    in FEATURE_SETS.items()
}


print()
print("=" * 110)
print("FINAL TOURNAMENT COMPLETE")
print("=" * 110)

print(
    "Saved:"
)

print(
    "FINAL_TOURNAMENT_ANNUAL"
)

print(
    "FINAL_TOURNAMENT_TRADES"
)

print(
    "FINAL_TOURNAMENT_DEVELOPMENT"
)

print(
    "FINAL_TOURNAMENT_CONFIRMATION"
)

print(
    "FINAL_TOURNAMENT_COST"
)

print(
    "FINAL_TOURNAMENT_LOO"
)

print(
    "FINAL_TOURNAMENT_SENSITIVITY"
)

print(
    "FINAL_TOURNAMENT_BOOTSTRAP"
)

print(
    "FINAL_TOURNAMENT_COMPARISON"
)

print(
    "FINAL_TOURNAMENT_QUALIFICATION"
)

print(
    "FINAL_TOURNAMENT_DECISION"
)
